<a href="https://colab.research.google.com/github/rushab14/A2AProject/blob/master/DB_vs_DB_less_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. INSTALL DEPENDENCIES
!pip install pypdf sentence-transformers numpy scikit-learn -q

import numpy as np
import time
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

# 2. INITIALIZE MODEL (Loading into RAM)
print("Loading Embedding Model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. UPLOAD PDF
print("\nStep 1: Upload your PDF file")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

# 4. PROCESS PDF (Vector-DB-less Logic)
def process_pdf_local(path, chunk_size=600):
    reader = PdfReader(path)
    text = ""
    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content + " "

    # Split text into manageable chunks
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    return chunks

print(f"\nStep 2: Chunking and Embedding '{pdf_filename}'...")
chunks = process_pdf_local(pdf_filename)

# Create the "Search Index" entirely in RAM
start_embed = time.time()
chunk_embeddings = model.encode(chunks)
end_embed = time.time()

print(f"Done! Created {len(chunks)} chunks in {end_embed - start_embed:.2f} seconds.")

# 5. PERFORM SEARCH
query = input("\nStep 3: Enter your question about the PDF: ")

start_search = time.time()
query_embedding = model.encode([query])

# Calculate similarity against every chunk (Exact Search)
distances = cosine_similarity(query_embedding, chunk_embeddings).flatten()
best_index = np.argmax(distances)
end_search = time.time()

# 6. RESULTS & METRICS
print("\n" + "="*50)
print(f"SEARCH RESULT (Confidence: {distances[best_index]:.4f})")
print("="*50)
print(chunks[best_index])
print("="*50)
print(f"Search Latency: {(end_search - start_search)*1000:.2f} ms")
print("Note: This search was performed in-memory without a database server.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.2 MB/s eta 0:00:00
Loading Embedding Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Step 1: Upload your PDF file


Saving wwf_tigers_e_1.pdf to wwf_tigers_e_1.pdf

Step 2: Chunking and Embedding 'wwf_tigers_e_1.pdf'...
Done! Created 26 chunks in 2.08 seconds.

Step 3: Enter your question about the PDF: how many sub species of tiger are there 

SEARCH RESULT (Confidence: 0.7652)
Species fact sheet:  
Tigers 
The largest cat of all, the tiger is a powerful symbol among 
the different cultures that share its home. But this 
magnificent animal is being persecuted across its range. 
Tigers are poisoned, shot, trapped, and snared, largely as a 
result of conflicts with people and to meet the demands of a 
continuing illegal trade in tiger derivatives and parts. On top 
of this, both their habitat and natural prey continue to 
disappear. Over the past 100 years, tiger numbers have 
declined by 95 per cent and three sub-species have become 
extinct — with a fourth not seen i
Search Latency: 17.68 ms
Note: This search was performed in-memory without a database server.
